In [31]:
import os
import json
import re
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from google import genai

from dotenv import load_dotenv

# ============================================================
# 1. Gemini client
# ============================================================

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY is not set")

client = genai.Client(api_key=api_key)

# ============================================================
# 2. Prompt
# ============================================================

INTRA_SYSTEM_PROMPT_SINGLE_PAIR = """
You are a specialized requirements engineering assistant for agile user stories. Your task is to classify the semantic relation between two actions (Action A and Action B) extracted from a single user story.

Base the classification ONLY on the provided story text and action pairs. You must choose exactly one label from: ENABLES, BLOCKS, NONE.

### LABEL DEFINITIONS & EXCLUSIONS

ENABLES
- Definition: One action establishes a state, condition, authorization, verification, setup, resource, or prior result that the other action requires in order to be executed.
- Rules: Use ENABLES when the source action functions as a necessary precondition for the target action. Use ENABLES even if the relationsihip is not direct, but one requires the other. 
- EXCLUSION: Do not assign ENABLES merely based on temporal order, sequence, or general helpfulness. (e.g., reading a page before clicking a button is temporal sequence, not a functional requirement).

BLOCKS
- Definition: One action creates, maintains, or implies a state, restriction, validation outcome, or rule that prevents the other action from being carried out.
- Rules: Use BLOCKS when the story dictates a genuine preventing relation, negative constraint, revocation of access, or mutually exclusive state.
- EXCLUSION: Do not use BLOCKS merely because actions represent alternate paths or differences in behavior. 
- EXCLUSION (Avoided Effort): Do NOT use BLOCKS when one action simply allows the user to avoid, skip, or bypass doing something (e.g., "doing X so I don't have to do Y"). Removing the need for a user to do something is a convenience, not a blocking constraint.

NONE
- Definition: Neither ENABLES nor BLOCKS is sufficiently supported by the text alone.
- Rules: Use NONE when actions are parallel, loosely related, or merely sequential without strict dependency. Also use NONE for actions that describe co-occurring architectural details, background states, or implementation locations rather than a sequential enabling/blocking relationship.
Prefer NONE over speculative assignments when the evidence is weak or ambiguous.

### DECISION RULES
1. Grounding: Base the decision ONLY on the wording of the user story and the two action descriptions. Do not infer hidden domain assumptions.
2. Step-by-Step Logic:  First, check if one action ENABLES the other based on the descriptions above. Second, check if one action BLOCKS the other based on the descriptions above. 
3. Output formatting: If the label is ENABLES or BLOCKS, set "source_action" to the action that DOES the enabling/blocking, and "target_action" to the action that IS enabled/blocked. If the label is NONE, set both direction fields to null.

### EXAMPLES

Example 1:
Story: As a Team Lead, I want to revoke a user's write access, so that they can no longer edit the project files.
Action A: revoke
Action B: edit
Output:
{
  "reason": "The story explicitly states that revoking access (Action A) removes the ability to edit the files (Action B). Therefore, Action A creates a restriction that prevents Action B.",
  "source_action": "A",
  "target_action": "B",
  "label": "BLOCKS"
}

Example 2:
Story: As a Content Creator, I want to upload a video, so that I can broadcast it to my subscribers.
Action A: upload
Action B: broadcast
Output:
{
  "reason": "Broadcasting the video (Action B) cannot occur unless the video has been made available to the system first through uploading (Action A). Uploading acts as the necessary precondition.",
  "source_action": "A",
  "target_action": "B",
  "label": "ENABLES"
}

Example 3:
Story: As a Developer, I want to deploy an application where the logs are saved on a secondary drive but the main app runs in memory, so that performance is not impacted.
Action A: deploy
Action B: save
Output:
{
  "reason": "Saving logs on a secondary drive is described as an architectural detail or background state of how the deployment happens. Deploying does not functionally enable or block the act of saving in a sequential user workflow; they are simply related concepts in the story.",
  "source_action": null,
  "target_action": null,
  "label": "NONE"
}

Example 4:
Story: As a System Admin, I want to manage the network with the new CLI tool, so that I can automate my workflow using standard commands.
Action A: manage
Action B: using
Output:
{
  "reason": "These actions represent loosely connected phrasing and descriptive grammar within the sentence rather than distinct, sequential system events. They are not logically connected in a way where one acts as a functional prerequisite or restriction for the other.",
  "source_action": null,
  "target_action": null,
  "label": "NONE"
}

### OUTPUT FORMAT
Return only valid JSON in this format:
{
  "results": [
    {
      "pair_index": 1,
      "input_action_a": "string",
      "input_action_b": "string",
      "source_action": "string|null",
      "target_action": "string|null",
      "label": "ENABLES|BLOCKS|NONE",
      "reason": "max 20 words"
    }
  ]
}
""".strip()


# ============================================================
# 3. Utility functions
# ============================================================

def clean_story_text(text):
    """Remove leading #G...# marker from story text."""
    if text is None:
        return ""
    return re.sub(r"^\s*#G\d+#\s*", "", str(text)).strip()


def extract_first_json_object(text):
    """
    Extract the first JSON object from a model response.
    Useful when the model accidentally wraps JSON in markdown or extra text.
    """
    if not text:
        raise ValueError("Empty response text from model.")

    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response:\n{text}")

    return match.group(0)


def parse_json_response(raw_text):
    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError:
        json_text = extract_first_json_object(raw_text)
        parsed = json.loads(json_text)

    # Simple fix: Gemini sometimes returns {"results": [ {...} ]}
    # even though we asked for one flat object.
    if isinstance(parsed, dict) and "results" in parsed:
        results = parsed["results"]
        if isinstance(results, list) and len(results) == 1:
            parsed = results[0]

    return parsed 


def normalize_null(value):
    """Convert string 'null' to Python None."""
    if value is None:
        return None
    if isinstance(value, str) and value.strip().lower() == "null":
        return None
    return value


def normalize_label(label):
    """Normalize relation label."""
    label = str(label).strip().upper()
    if label not in {"ENABLES", "BLOCKS", "NONE"}:
        raise ValueError(f"Invalid label returned by model: {label}")
    return label


def derive_direction(input_action_a, input_action_b, source_action, target_action, label):
    """
    Derive A_TO_B or B_TO_A from source/target actions.
    Returns None for NONE or invalid direction.
    """
    if label not in {"ENABLES", "BLOCKS"}:
        return None

    if source_action == input_action_a and target_action == input_action_b:
        return "A_TO_B"

    if source_action == input_action_b and target_action == input_action_a:
        return "B_TO_A"

    return None


def build_single_pair_user_prompt(story_dict, pair_index, action_a, action_b):
    """Build prompt for exactly one story and one action pair."""
    return f"""
Classify exactly one action pair in this user story.

Story index: {story_dict["story_index"]}
Pair index: {pair_index}

User story:
{story_dict["Text"]}

Action pair:
Action A="{action_a}"
Action B="{action_b}"
""".strip()


def validate_single_pair_output(parsed):
    """Validate that the model returned the expected fields."""
    required_fields = {
        "pair_index",
        "input_action_a",
        "input_action_b",
        "source_action",
        "target_action",
        "label",
        "reason",
    }

    missing = required_fields - set(parsed.keys())
    if missing:
        raise ValueError(f"JSON missing fields: {missing}. Full JSON: {parsed}")

    label = normalize_label(parsed["label"])

    source_action = normalize_null(parsed.get("source_action"))
    target_action = normalize_null(parsed.get("target_action"))

    if label == "NONE":
        if source_action is not None or target_action is not None:
            raise ValueError(
                "For label NONE, source_action and target_action must be null. "
                f"Full JSON: {parsed}"
            )

    if label in {"ENABLES", "BLOCKS"}:
        if source_action is None or target_action is None:
            raise ValueError(
                f"For label {label}, source_action and target_action must not be null. "
                f"Full JSON: {parsed}"
            )

    return True


# ============================================================
# 4. Single-pair Gemini call
# ============================================================

def classify_single_intra_pair_gemini(
    story_dict,
    pair_index,
    action_a,
    action_b,
    model_name="gemini-3.1-flash-lite-preview",
    max_retries=3,
    retry_sleep_sec=2,
):
    """
    One API call for exactly:
        one story + one action pair

    Returns:
        {
            "parsed": parsed_json,
            "raw_response": raw_text,
            "attempts": int
        }
    """
    user_prompt = build_single_pair_user_prompt(
        story_dict=story_dict,
        pair_index=pair_index,
        action_a=action_a,
        action_b=action_b,
    )

    full_prompt = f"{INTRA_SYSTEM_PROMPT_SINGLE_PAIR}\n\n{user_prompt}"

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=full_prompt,
            )

            raw_text = response.text
            parsed = parse_json_response(raw_text)

            validate_single_pair_output(parsed)

            return {
                "parsed": parsed,
                "raw_response": raw_text,
                "attempts": attempt,
            }

        except Exception as e:
            last_error = e

            if attempt < max_retries:
                time.sleep(retry_sleep_sec * attempt)
            else:
                raise RuntimeError(
                    f"Failed after {max_retries} attempts for "
                    f"story_index={story_dict.get('story_index')}, "
                    f"pair_index={pair_index}. Last error: {last_error}"
                )


# ============================================================
# 5. Convert one model response to one dataframe row
# ============================================================

def single_pair_result_to_row(
    story_dict,
    pair_index,
    action_a,
    action_b,
    llm_output,
    response_time_sec,
):
    parsed = llm_output["parsed"]
    raw_response = llm_output["raw_response"]

    input_action_a = parsed.get("input_action_a")
    input_action_b = parsed.get("input_action_b")

    source_action = normalize_null(parsed.get("source_action"))
    target_action = normalize_null(parsed.get("target_action"))

    if source_action == "A":
        source_action = action_a
    elif source_action == "B":
        source_action = action_b
    
    if target_action == "A":
        target_action = action_a
    elif target_action == "B":
        target_action = action_b


    label = normalize_label(parsed.get("label"))
    reason = parsed.get("reason", "")

    relation_direction = derive_direction(
        input_action_a=input_action_a,
        input_action_b=input_action_b,
        source_action=source_action,
        target_action=target_action,
        label=label,
    )

    return {
        "story_index": story_dict["story_index"],
        "story_text": story_dict["Text"],
        "pair_index": pair_index,

        # Original pair from your input data
        "original_action_a": action_a,
        "original_action_b": action_b,

        # Actions as returned by the model
        "input_action_a": input_action_a,
        "input_action_b": input_action_b,

        "predicted_relation": label,
        "source_action": source_action,
        "target_action": target_action,
        "relation_direction": relation_direction,
        "llm_reason": reason,

        "response_time_sec": response_time_sec,
        "attempts": llm_output.get("attempts"),
        "raw_response": raw_response,
        "status": "OK",
        "error": None,
    }


def failed_pair_to_row(
    story_dict,
    pair_index,
    action_a,
    action_b,
    response_time_sec,
    error,
):
    """Create a row even when a pair fails."""
    return {
        "story_index": story_dict["story_index"],
        "story_text": story_dict["Text"],
        "pair_index": pair_index,

        "original_action_a": action_a,
        "original_action_b": action_b,

        "input_action_a": action_a,
        "input_action_b": action_b,

        "predicted_relation": None,
        "source_action": None,
        "target_action": None,
        "relation_direction": None,
        "llm_reason": None,

        "response_time_sec": response_time_sec,
        "attempts": None,
        "raw_response": None,
        "status": "ERROR",
        "error": str(error),
    }


# ============================================================
# 6. Main experiment runner
# ============================================================

def run_intra_story_experiment_gemini_single_pair_calls(
    stories,
    n_stories=None,
    model_name="gemini-3.1-flash-lite-preview",
    max_retries=3,
    retry_sleep_sec=2,
    save_every_n_pairs=None,
    output_csv_path=None,
):
    """
    Runs intra-story labeling with separate API calls.

    Processing unit:
        one story + one action pair = one Gemini API call

    Args:
        stories:
            List of story dictionaries.
            Each story must contain:
                - story_index
                - Text
                - action_pairs

        n_stories:
            Optional limit for testing.

        model_name:
            Gemini model name.

        max_retries:
            Number of retries per pair.

        retry_sleep_sec:
            Base sleep time between retries.

        save_every_n_pairs:
            Optional checkpoint interval.
            Example: 50 saves results every 50 processed pairs.

        output_csv_path:
            Optional CSV path for checkpoint/final results.

    Returns:
        results_df, summary_df, summary
    """
    run_stories = stories[:n_stories] if n_stories is not None else stories

    all_rows = []
    summary_rows = []

    run_start = time.perf_counter()
    processed_pairs = 0

    total_pairs = sum(len(story.get("action_pairs", [])) for story in run_stories)

    progress_bar = tqdm(total=total_pairs, desc="Running single-pair intra-story labeling")

    for story_dict in run_stories:
        story_index = story_dict.get("story_index")
        action_pairs = story_dict.get("action_pairs", [])

        story_start = time.perf_counter()
        story_ok = 0
        story_error = 0

        print(
            f"Starting story_index={story_index} "
            f"with {len(action_pairs)} separate pair calls..."
        )

        for pair_index, pair in enumerate(action_pairs, start=1):
            pair_start = time.perf_counter()

            try:
                if not isinstance(pair, (list, tuple)) or len(pair) != 2:
                    raise ValueError(f"Invalid action pair format: {pair}")

                action_a, action_b = pair

                llm_output = classify_single_intra_pair_gemini(
                    story_dict=story_dict,
                    pair_index=pair_index,
                    action_a=action_a,
                    action_b=action_b,
                    model_name=model_name,
                    max_retries=max_retries,
                    retry_sleep_sec=retry_sleep_sec,
                )

                pair_time = time.perf_counter() - pair_start

                row = single_pair_result_to_row(
                    story_dict=story_dict,
                    pair_index=pair_index,
                    action_a=action_a,
                    action_b=action_b,
                    llm_output=llm_output,
                    response_time_sec=pair_time,
                )

                story_ok += 1

            except Exception as e:
                pair_time = time.perf_counter() - pair_start

                row = failed_pair_to_row(
                    story_dict=story_dict,
                    pair_index=pair_index,
                    action_a=pair[0] if isinstance(pair, (list, tuple)) and len(pair) > 0 else None,
                    action_b=pair[1] if isinstance(pair, (list, tuple)) and len(pair) > 1 else None,
                    response_time_sec=pair_time,
                    error=e,
                )

                story_error += 1
                print(
                    f"ERROR story_index={story_index}, "
                    f"pair_index={pair_index}: {e}"
                )

            all_rows.append(row)
            processed_pairs += 1
            progress_bar.update(1)

            if (
                save_every_n_pairs is not None
                and output_csv_path is not None
                and processed_pairs % save_every_n_pairs == 0
            ):
                checkpoint_df = pd.DataFrame(all_rows)
                checkpoint_df.to_csv(output_csv_path, index=False, encoding="utf-8")
                print(f"Checkpoint saved after {processed_pairs} pairs: {output_csv_path}")

        story_time = time.perf_counter() - story_start

        summary_rows.append({
            "story_index": story_index,
            "n_action_pairs": len(action_pairs),
            "n_ok": story_ok,
            "n_error": story_error,
            "response_time_sec": story_time,
            "status": "OK" if story_error == 0 else "PARTIAL_ERROR",
        })

        print(
            f"Finished story_index={story_index}: "
            f"{story_ok} OK, {story_error} errors, {story_time:.2f}s"
        )

    progress_bar.close()

    total_time = time.perf_counter() - run_start

    results_df = pd.DataFrame(all_rows)
    summary_df = pd.DataFrame(summary_rows)

    if output_csv_path is not None:
        results_df.to_csv(output_csv_path, index=False, encoding="utf-8")
        print(f"Final results saved to: {output_csv_path}")

    summary = {
        "n_stories": len(run_stories),
        "n_pairs_total": total_pairs,
        "n_pairs_processed": len(results_df),
        "n_pairs_ok": int((results_df["status"] == "OK").sum()) if not results_df.empty else 0,
        "n_pairs_error": int((results_df["status"] == "ERROR").sum()) if not results_df.empty else 0,
        "total_time_sec": total_time,
        "avg_time_per_pair_sec": results_df["response_time_sec"].mean() if not results_df.empty else None,
        "avg_time_per_story_sec": summary_df["response_time_sec"].mean() if not summary_df.empty else None,
    }

    print(summary)

    return results_df, summary_df, summary


# ============================================================
# 7. Load data
# ============================================================

input_path = Path(r"preprocessed_pairs\g14_turbo_30.json")

with input_path.open("r", encoding="utf-8") as f:
    intra_stories = json.load(f)

for story in intra_stories:
    story["Text"] = clean_story_text(story.get("Text", ""))


# ============================================================
# 8. Test run: first 3 stories
# ============================================================

intra_results_3, intra_summary_3_df, intra_summary_3 = (
    run_intra_story_experiment_gemini_single_pair_calls(
        stories=intra_stories,
        n_stories=3,
        model_name="gemini-3.1-flash-lite-preview",
        max_retries=3,
        retry_sleep_sec=2,
        save_every_n_pairs=10,
    )
)

display(intra_results_3.head())
display(intra_summary_3_df)


Running single-pair intra-story labeling:   0%|          | 0/3 [00:00<?, ?it/s]

Starting story_index=0 with 1 separate pair calls...
Finished story_index=0: 1 OK, 0 errors, 0.91s
Starting story_index=1 with 1 separate pair calls...
Finished story_index=1: 1 OK, 0 errors, 0.74s
Starting story_index=2 with 1 separate pair calls...
Finished story_index=2: 1 OK, 0 errors, 0.72s
{'n_stories': 3, 'n_pairs_total': 3, 'n_pairs_processed': 3, 'n_pairs_ok': 3, 'n_pairs_error': 0, 'total_time_sec': 2.3839317000238225, 'avg_time_per_pair_sec': np.float64(0.789035233280932), 'avg_time_per_story_sec': np.float64(0.7903755333196992)}


,story_index,story_text,pair_index,original_action_a,original_action_b,input_action_a,input_action_b,predicted_relation,source_action,target_action,relation_direction,llm_reason,response_time_sec,attempts,raw_response,status,error
0,0,"As a Publisher, I want to publish a dataset, s...",1,publish,view,publish,view,ENABLES,publish,view,A_TO_B,Publishing the dataset is a necessary precondi...,0.907844,1,"{\n ""results"": [\n {\n ""pair_index"": ...",OK,None
1,1,"As a Publisher, I want to publish a dataset, s...",1,publish,share,publish,share,ENABLES,publish,share,A_TO_B,Publishing the dataset is a necessary conditio...,0.739902,1,"{\n ""results"": [\n {\n ""pair_index"": ...",OK,None
2,2,"As a Publisher, I want to sign up for an accou...",1,sign up,publish,sign up,publish,ENABLES,sign up,publish,A_TO_B,Signing up creates the required account author...,0.719359,1,"{\n ""results"": [\n {\n ""pair_index"": ...",OK,None


,story_index,n_action_pairs,n_ok,n_error,response_time_sec,status
0,0,1,1,0,0.908850,OK
1,1,1,1,0,0.741219,OK
2,2,1,1,0,0.721058,OK


In [33]:


# ============================================================
# 9. Full run
# ============================================================

intra_results_full, intra_summary_full_df, intra_summary_full = (
    run_intra_story_experiment_gemini_single_pair_calls(
        stories=intra_stories,
        n_stories=None,
        model_name="gemini-3.1-flash-lite-preview",
        max_retries=3,
        retry_sleep_sec=2,
        save_every_n_pairs=50,
        output_csv_path=r"pred_res\neo4j\g14_intra_story_30.csv",
    )
)

display(intra_results_full.head())
display(intra_summary_full_df)

Running single-pair intra-story labeling:   0%|          | 0/53 [00:00<?, ?it/s]

Starting story_index=0 with 1 separate pair calls...
Finished story_index=0: 1 OK, 0 errors, 0.84s
Starting story_index=1 with 1 separate pair calls...
Finished story_index=1: 1 OK, 0 errors, 0.72s
Starting story_index=2 with 1 separate pair calls...
Finished story_index=2: 1 OK, 0 errors, 0.68s
Starting story_index=7 with 1 separate pair calls...
Finished story_index=7: 1 OK, 0 errors, 0.85s
Starting story_index=10 with 3 separate pair calls...
Finished story_index=10: 3 OK, 0 errors, 2.46s
Starting story_index=11 with 3 separate pair calls...
Finished story_index=11: 3 OK, 0 errors, 2.66s
Starting story_index=16 with 3 separate pair calls...
Finished story_index=16: 3 OK, 0 errors, 2.46s
Starting story_index=17 with 1 separate pair calls...
Finished story_index=17: 1 OK, 0 errors, 0.82s
Starting story_index=18 with 3 separate pair calls...
Finished story_index=18: 3 OK, 0 errors, 2.35s
Starting story_index=19 with 1 separate pair calls...
Finished story_index=19: 1 OK, 0 errors, 0.72

,story_index,story_text,pair_index,original_action_a,original_action_b,input_action_a,input_action_b,predicted_relation,source_action,target_action,relation_direction,llm_reason,response_time_sec,attempts,raw_response,status,error
0,0,"As a Publisher, I want to publish a dataset, s...",1,publish,view,publish,view,ENABLES,publish,view,A_TO_B,Publishing the dataset is a necessary precondi...,0.835681,1,"{\n ""results"": [\n {\n ""pair_index"": ...",OK,None
1,1,"As a Publisher, I want to publish a dataset, s...",1,publish,share,publish,share,ENABLES,publish,share,A_TO_B,Publishing the dataset is a necessary conditio...,0.716432,1,"{\n ""results"": [\n {\n ""pair_index"": ...",OK,None
2,2,"As a Publisher, I want to sign up for an accou...",1,sign up,publish,sign up,publish,ENABLES,sign up,publish,A_TO_B,Signing up creates the publisher account neces...,0.678296,1,"{\n ""results"": [\n {\n ""pair_index"": ...",OK,None
3,7,"As a Publisher, I want to use a publish comman...",1,use,update,use,update,ENABLES,use,update,A_TO_B,The user must use the publish command (Action ...,0.850787,1,"{\n ""results"": [\n {\n ""pair_index"": ...",OK,None
4,10,"As a Publisher, I want to validate the data I ...",1,validate,publish,validate,publish,ENABLES,validate,publish,A_TO_B,Validation is a necessary precondition to ensu...,0.760690,1,"{\n ""results"": [\n {\n ""pair_index"": ...",OK,None


,story_index,n_action_pairs,n_ok,n_error,response_time_sec,status
0,0,1,1,0,0.837126,OK
1,1,1,1,0,0.718566,OK
2,2,1,1,0,0.680513,OK
3,7,1,1,0,0.852437,OK
4,10,3,3,0,2.463402,OK
5,11,3,3,0,2.655938,OK
6,16,3,3,0,2.458371,OK
7,17,1,1,0,0.817522,OK
8,18,3,3,0,2.348774,OK
9,19,1,1,0,0.722355,OK
